## **Exercisi 8**

Carga el conjunto de datos MNIST ( introducido en el capiítulo 3 ) y dividelo en un conjunto de entrenamiento, un conjunto de validación y un conjunto de prueba ( por ejemplo, utiliza 50.000 instancias para el entrenamiento, 10.000 para la validación y 10.0000 para la prueba). Después, entrena varios clasificadores, como un clasificador random forest, un clasificador extra -trees y un cladificador SVM. A cont9inuación, intenta combinarlos en un ensamble que supere el rendimiento a cada clasificador individual del conjunot de validación, utilizando soft voting o hard voting. Una vez que hayas encontrado uno, pruébalo en el conjunto de prueba. ¿Cuánto ha mejorado el rendimiento en comparación con los clasificadores individuales?

### Solución: selección mediante validación

Usamos 50.000 imágenes para entrenar, 10.000 para validar y las 10.000 imágenes finales de MNIST para probar. La división de entrenamiento/validación es aleatoria y estratificada. Normalizamos los píxeles a [0, 1] mediante una constante, sin aprender nada del conjunto de prueba.

Entrenamos Random Forest, Extra Trees y una SVM con kernel RBF. Probamos votación **hard** con distintos pesos, además de votación **soft** entre los dos bosques. La SVM no participa en soft voting porque no hemos ajustado un modelo de probabilidades. Reutilizamos las predicciones de los modelos ya entrenados: no es necesario volver a entrenarlos para cada combinación. La búsqueda es pequeña y se decide exclusivamente con validación.

In [1]:
import time
from itertools import product
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

SEED = 42
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(
    X[:60000], y[:60000], test_size=10000,
    stratify=y[:60000], random_state=SEED,
)
X_test, y_test = X[60000:], y[60000:]
assert (len(y_train), len(y_valid), len(y_test)) == (50000, 10000, 10000)
pd.DataFrame({"partición": ["entrenamiento", "validación", "prueba"],
              "imágenes": [len(y_train), len(y_valid), len(y_test)]})

,partición,imágenes
0,entrenamiento,50000
1,validación,10000
2,prueba,10000


In [2]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=4, random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, n_jobs=4, random_state=SEED),
    "SVM RBF": SVC(C=3.0, gamma="scale", cache_size=1024, random_state=SEED),
}
training_seconds = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds[name] = time.perf_counter() - start
    print(f"{name}: entrenado en {training_seconds[name]:.1f} s", flush=True)

classes = np.unique(y_train)
assert all(np.array_equal(m.classes_, classes) for m in models.values())
valid_predictions = np.column_stack([m.predict(X_valid) for m in models.values()])
valid_probas = np.stack([models[name].predict_proba(X_valid)
                         for name in ("Random Forest", "Extra Trees")])
valid_individual = pd.Series(
    [accuracy_score(y_valid, valid_predictions[:, i]) for i in range(len(models))],
    index=list(models), name="accuracy_validación",
)
display(pd.concat([valid_individual, pd.Series(training_seconds, name="segundos_entrenamiento")], axis=1))

Random Forest: entrenado en 8.9 s


Extra Trees: entrenado en 8.1 s


SVM RBF: entrenado en 101.4 s


,accuracy_validación,segundos_entrenamiento
Random Forest,0.9669,8.904252
Extra Trees,0.9709,8.119674
SVM RBF,0.9821,101.402803


In [3]:
def hard_vote(predictions, weights):
    # En caso de empate, argmax elige la primera clase (orden ascendente).
    counts = (predictions[:, :, None] == classes[None, None, :])
    scores = (counts * np.asarray(weights)[None, :, None]).sum(axis=1)
    return classes[scores.argmax(axis=1)]

def soft_vote(probas, weights):
    return classes[np.average(probas, axis=0, weights=weights).argmax(axis=1)]

candidates = []
# Pesos positivos: todos los modelos participan en cada hard vote.
for weights in product((1, 2, 3), repeat=3):
    pred = hard_vote(valid_predictions, weights)
    candidates.append({"tipo": "hard", "pesos": weights,
                       "accuracy_validación": accuracy_score(y_valid, pred)})
for weights in ((1, 1), (1, 2), (2, 1)):
    pred = soft_vote(valid_probas, weights)
    candidates.append({"tipo": "soft", "pesos": weights,
                       "accuracy_validación": accuracy_score(y_valid, pred)})
ranking = pd.DataFrame(candidates).sort_values(
    "accuracy_validación", ascending=False, kind="stable"
).reset_index(drop=True)
best = ranking.iloc[0]
display(ranking.head(10))
validation_gain = best["accuracy_validación"] - valid_individual.max()
print(f"Votación elegida: {best['tipo']}, pesos={best['pesos']}")
print(f"Diferencia frente al mejor individual en validación: {100 * validation_gain:+.3f} puntos porcentuales.")
if validation_gain <= 0:
    print("En esta búsqueda, la votación no supera al mejor modelo individual; no se presupone una mejora.")

,tipo,pesos,accuracy_validación
0,hard,"(1, 1, 3)",0.9821
1,hard,"(1, 1, 2)",0.9771
2,hard,"(1, 2, 3)",0.9771
3,hard,"(2, 1, 3)",0.9771
4,hard,"(2, 2, 3)",0.9731
5,hard,"(1, 2, 2)",0.9730
6,hard,"(1, 3, 3)",0.9730
7,hard,"(2, 3, 3)",0.9730
8,hard,"(3, 2, 2)",0.9730
9,hard,"(1, 1, 1)",0.9729


Votación elegida: hard, pesos=(1, 1, 3)
Diferencia frente al mejor individual en validación: +0.000 puntos porcentuales.
En esta búsqueda, la votación no supera al mejor modelo individual; no se presupone una mejora.


In [4]:
# La configuración ya está fijada: ahora se evalúa en prueba.
# Conservamos estas predicciones para el ejercicio 9.
test_predictions = np.column_stack([m.predict(X_test) for m in models.values()])
if best["tipo"] == "hard":
    voting_test_prediction = hard_vote(test_predictions, best["pesos"])
else:
    test_probas = np.stack([models[name].predict_proba(X_test)
                            for name in ("Random Forest", "Extra Trees")])
    voting_test_prediction = soft_vote(test_probas, best["pesos"])
voting_test_score = accuracy_score(y_test, voting_test_prediction)
individual_test_scores = [accuracy_score(y_test, test_predictions[:, i])
                          for i in range(len(models))]
comparison = pd.DataFrame({
    "accuracy_validación": [*valid_individual, best["accuracy_validación"]],
    "accuracy_prueba": [*individual_test_scores, voting_test_score],
}, index=[*models, "Votación elegida"])
comparison["errores_prueba"] = np.rint(
    (1 - comparison["accuracy_prueba"]) * len(y_test)
).astype(int)
display(comparison)
for name, score in zip(models, individual_test_scores):
    print(f"Votación frente a {name}: {100 * (voting_test_score - score):+.3f} puntos porcentuales "
          f"({int(round((voting_test_score - score) * len(y_test))):+d} aciertos).")

,accuracy_validación,accuracy_prueba,errores_prueba
Random Forest,0.9669,0.9698,302
Extra Trees,0.9709,0.9717,283
SVM RBF,0.9821,0.9819,181
Votación elegida,0.9821,0.9819,181


Votación frente a Random Forest: +1.210 puntos porcentuales (+121 aciertos).
Votación frente a Extra Trees: +1.020 puntos porcentuales (+102 aciertos).
Votación frente a SVM RBF: +0.000 puntos porcentuales (+0 aciertos).


**Interpretación.** La tabla y las diferencias anteriores responden cuánto mejora (o empeora) la votación frente a cada clasificador. Un ensamble puede reducir errores si sus componentes se equivocan en imágenes distintas, pero no está garantizado que supere al mejor modelo. La selección sobre varias combinaciones hace que el resultado de validación pueda ser optimista; el conjunto de prueba proporciona la comparación final. No ajustamos pesos a partir de estos resultados.

## **Exercisi 9**

Ejecuta lso clasificadores individuales del ejercicio anterior para hacer predicciones en el conjunto de validación y crea un nuevo conjunto de entrenamiento con las predicciones resultantes: cada instancia de entrenamiento es un vector que conteine el conjunto de predicciones de todos tus clasificadores para una imagen y el objetivo es la clase de la imagen. Entrena un clasificador en este nuevo conjunto de entrenamiento. ¡Enhorabuena, king! Aora, evalua el ensamble en el conjunto de prueba. Para cada imagen del conjunto de prueba, haz predicciones en el blender para obtener las predicciones del ensamble. ¿Cómo es en comparación con el clasificador de votación que has entrenado antes?

### Solución: un segundo nivel que aprende a combinar

Cada fila del nuevo conjunto contiene las **tres clases predichas** por los modelos base para una imagen de validación; su objetivo es el dígito real. Esto es *blending*, una variante de stacking con una partición reservada. Los modelos base nunca se entrenaron con estas imágenes.

Usamos codificación one-hot y regresión logística: los números de las clases son categorías, no magnitudes (un 8 no es «el doble» de un 4). El blender aprende qué predicciones de cada modelo apoyan cada clase. Sus hiperparámetros se fijan de antemano. No reentrenamos los modelos base con validación, ya que cambiaría el proceso que produce sus entradas.

In [5]:
X_blend_train = valid_predictions
X_blend_test = test_predictions
assert X_blend_train.shape == (10000, 3)
assert X_blend_test.shape == (10000, 3)
display(pd.DataFrame(X_blend_train[:10], columns=list(models)).assign(real=y_valid[:10]))

blender = make_pipeline(
    OneHotEncoder(categories=[classes] * len(models), handle_unknown="ignore"),
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
)
blender.fit(X_blend_train, y_valid)
blended_prediction = blender.predict(X_blend_test)
blended_test_score = accuracy_score(y_test, blended_prediction)

final_comparison = pd.DataFrame({
    "accuracy_prueba": [voting_test_score, blended_test_score],
    "errores_prueba": [np.count_nonzero(voting_test_prediction != y_test),
                       np.count_nonzero(blended_prediction != y_test)],
}, index=["Votación elegida", "Blending"])
display(final_comparison)
difference = blended_test_score - voting_test_score
print(f"Blending frente a votación: {100 * difference:+.3f} puntos porcentuales.")
print(f"Diferencia de aciertos: {int(round(difference * len(y_test))):+d} de {len(y_test)}.")
print("Blending supera a votación." if difference > 0 else
      "Ambos empatan en accuracy." if difference == 0 else
      "La votación obtiene mejor accuracy que blending.")

,Random Forest,Extra Trees,SVM RBF,real
0,7,7,7,7
1,8,8,8,8
2,4,4,4,4
3,8,8,8,8
4,2,2,2,2
5,9,9,9,9
6,6,6,6,6
7,8,8,8,8
8,6,6,6,6
9,5,5,5,5


,accuracy_prueba,errores_prueba
Votación elegida,0.9819,181
Blending,0.9814,186


Blending frente a votación: -0.050 puntos porcentuales.
Diferencia de aciertos: -5 de 10000.
La votación obtiene mejor accuracy que blending.


**Conclusión metodológica.** La salida anterior compara ambos ensambles sobre las mismas imágenes de prueba. No evaluamos el blender sobre validación como si fuera una estimación independiente: esa partición ahora es su conjunto de entrenamiento. El one-hot evita imponer un orden artificial a los dígitos, aunque las etiquetas predichas descartan la confianza de cada modelo. Una extensión sería usar probabilidades como características, o stacking con predicciones out-of-fold, pero no se utilizan aquí para cambiar el experimento tras ver el test. Diferencias pequeñas de accuracy en una sola partición no demuestran por sí solas una mejora estadísticamente significativa.

### Resultados de esta ejecución

| Modelo | Accuracy en prueba | Errores (10.000 imágenes) |
|---|---:|---:|
| Random Forest | 96,98 % | 302 |
| Extra Trees | 97,17 % | 283 |
| SVM RBF | 98,19 % | 181 |
| Votación seleccionada | 98,19 % | 181 |
| Blending | 98,14 % | 186 |

**Ejercicio 8.** La mejor configuración en validación fue hard voting con pesos `(1, 1, 3)`, en el orden Random Forest, Extra Trees y SVM. Su accuracy de validación fue 98,21 %, igual que la SVM. Es un caso degenerado: el peso 3 de la SVM supera la suma de los otros dos pesos, así que la votación reproduce exactamente sus predicciones. Por tanto, **no hemos encontrado una combinación que supere a todos los clasificadores individuales**. En prueba mejora al Random Forest en 1,21 puntos porcentuales y a Extra Trees en 1,02 puntos, pero no mejora a la SVM. Esto ilustra el límite de esta búsqueda; no constituye una ventaja de combinar modelos. Las combinaciones en las que los bosques sí pueden cambiar la decisión de la SVM obtuvieron peores resultados de validación.

**Ejercicio 9.** El blender obtiene 98,14 % en prueba: 0,05 puntos porcentuales menos que la votación (5 errores adicionales). Aprender la combinación no garantizó una mejora. No hemos modificado los modelos ni los pesos después de observar el test.
